In [1]:
"""
phase1_priors.py — Expert Prior Specification & Parameter Store
===============================================================
This is the ENTRY POINT of the pipeline. It encodes expert knowledge
about each risk source as Beta distribution parameters and stores them
in a versioned JSON parameter store.

No labeled data is required at this stage.

Beta(alpha, beta) intuition:
  - alpha / (alpha + beta) = mean (your estimated base rate)
  - alpha + beta = concentration (your confidence / effective sample size)

Run:
    python phase1_priors.py
Output:
    params/param_store.json
"""

'\nphase1_priors.py — Expert Prior Specification & Parameter Store\n===============================================================\nThis is the ENTRY POINT of the pipeline. It encodes expert knowledge\nabout each risk source as Beta distribution parameters and stores them\nin a versioned JSON parameter store.\n\nNo labeled data is required at this stage.\n\nBeta(alpha, beta) intuition:\n  - alpha / (alpha + beta) = mean (your estimated base rate)\n  - alpha + beta = concentration (your confidence / effective sample size)\n\nRun:\n    python phase1_priors.py\nOutput:\n    params/param_store.json\n'

In [2]:
import json
import os
import numpy as np
from datetime import datetime, timezone
import pandas as pd
from typing import Any
import warnings

In [3]:
# ---------------------------------------------------------------------------
# 1. Expert knowledge: (mean_risk, concentration)
#    concentration ~ "how many virtual observations" back up your estimate
#    Low  (10–30):  wide uncertainty, e.g. novel source with sparse data
#    Med  (50–200): moderate certainty, e.g. source validated on past cases
#    High (300+):   tight prior, e.g. well-studied source with rich history
# ---------------------------------------------------------------------------
SOURCE_SPECS: dict[str, tuple[float, float]] = {
    "S1": (0.05,  40),   # Very low risk, lower confidence
    "S2": (0.15,   20),   # Low risk,  low confidence (novel source)
    "S3": (0.35,  100),   # Moderate risk
    "S4": (0.85,   200),   # High risk, very uncertain (sparse evidence)
    "S5": (0.45,  60),   # Low-moderate, solid evidence
    "S6": (0.20,   40),   # Low, limited data
    "S7": (0.95,  300),   # Very High risk, strong evidence
    "S8": (0.01, 1000),   # Tiny risk, very well validated
}

LEAK_SPEC = (0.02, 100)   # ~2% baseline unexplained risk, confident

In [4]:
def mean_conc_to_alpha_beta(mean: float, concentration: float) -> tuple[float, float]:
    """
    Convert (mean, concentration) → (alpha, beta).
    Floor at 1.1 prevents degenerate Beta shapes (U-shaped or half-bathtub).
    """
    alpha = max(mean * concentration, 1.1)
    beta  = max((1.0 - mean) * concentration, 1.1)
    return float(alpha), float(beta)


def alpha_beta_to_mean_conc(alpha: float, beta: float) -> tuple[float, float]:
    conc = alpha + beta
    mean = alpha / conc
    return float(mean), float(conc)


def beta_stats(alpha: float, beta: float) -> dict:
    """Return key statistics of a Beta(alpha, beta) distribution."""
    mean = alpha / (alpha + beta)
    variance = (alpha * beta) / ((alpha + beta) ** 2 * (alpha + beta + 1))
    std = variance ** 0.5
    # 90% credible interval via normal approximation (fast, good enough for diagnostics)
    z = 1.645
    lo = max(0.0, mean - z * std)
    hi = min(1.0, mean + z * std)
    return {
        "mean":  round(mean, 6),
        "std":   round(std, 6),
        "ci90_lo": round(lo, 6),
        "ci90_hi": round(hi, 6),
    }


def build_param_store(
    source_specs: dict[str, tuple[float, float]],
    leak_spec: tuple[float, float],
    version: str | None = None,
) -> dict:
    """
    Build a versioned parameter store dict from human-readable specs.
    """
    if version is None:
        version = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

    sources = {}
    for name, (mean, conc) in source_specs.items():
        alpha, beta = mean_conc_to_alpha_beta(mean, conc)
        sources[name] = {
            "alpha": alpha,
            "beta":  beta,
            **beta_stats(alpha, beta),
            "expert_mean": mean,
            "expert_concentration": conc,
        }

    leak_alpha, leak_beta = mean_conc_to_alpha_beta(*leak_spec)
    leak = {
        "alpha": leak_alpha,
        "beta":  leak_beta,
        **beta_stats(leak_alpha, leak_beta),
        "expert_mean": leak_spec[0],
        "expert_concentration": leak_spec[1],
    }

    return {
        "version": version,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_order": list(source_specs.keys()),
        "sources": sources,
        "leak": leak,
    }


def save_param_store(store: dict, path: str = "params/param_store.json") -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(store, f, indent=4)
    print(f"[phase1] Parameter store saved → {path}  (version: {store['version']})")


def load_param_store(path: str = "params/param_store.json") -> dict:
    with open(path) as f:
        return json.load(f)


def bayesian_update_source(
    store: dict,
    source_name: str,
    n_successes: int,
    n_trials: int,
) -> dict:
    """
    Conjugate Beta update for a single source given new Bernoulli observations.
    Use this when you accumulate partial labels over time (e.g., analyst reviews).

    updated_alpha = prior_alpha + n_successes
    updated_beta  = prior_beta  + (n_trials - n_successes)

    This is mathematically exact — no MCMC needed.
    """
    src = store["sources"][source_name]
    new_alpha = src["alpha"] + n_successes
    new_beta  = src["beta"]  + (n_trials - n_successes)

    updated = {
        "alpha": new_alpha,
        "beta":  new_beta,
        **beta_stats(new_alpha, new_beta),
        "expert_mean": src["expert_mean"],
        "expert_concentration": src["expert_concentration"],
        "update_note": f"Conjugate update: +{n_successes} successes / {n_trials} trials",
    }
    store["sources"][source_name] = updated
    store["version"] = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_updated"
    return store

In [5]:
# ---------------------------------------------------------------------------
# Diagnostic printout
# ---------------------------------------------------------------------------
def print_prior_summary(store: dict) -> None:
    print("\n" + "=" * 65)
    print(f"  Parameter Store — version: {store['version']}")
    print("=" * 65)
    print(f"  {'Source':<8} {'Mean':>8} {'Std':>8} {'CI90_lo':>9} {'CI90_hi':>9}")
    print("-" * 65)
    for name, s in store["sources"].items():
        print(f"  {name:<8} {s['mean']:>8.4f} {s['std']:>8.4f} "
              f"{s['ci90_lo']:>9.4f} {s['ci90_hi']:>9.4f}")
    lk = store["leak"]
    print(f"  {'Leak':<8} {lk['mean']:>8.4f} {lk['std']:>8.4f} "
          f"{lk['ci90_lo']:>9.4f} {lk['ci90_hi']:>9.4f}")
    print("=" * 65 + "\n")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    store = build_param_store(SOURCE_SPECS, LEAK_SPEC)
    print_prior_summary(store)
    save_param_store(store)

    # --- Example: update S2 after 10 analyst reviews (3 confirmed risk events)
    store_updated = bayesian_update_source(store, "S2", n_successes=3, n_trials=10)
    print("S2 after conjugate update:")
    s2 = store_updated["sources"]["S2"]
    print(f"  mean={s2['mean']:.4f}  std={s2['std']:.4f}  "
          f"(was expert_mean={s2['expert_mean']:.4f})\n")


  Parameter Store — version: 20260221_175538
  Source       Mean      Std   CI90_lo   CI90_hi
-----------------------------------------------------------------
  S1         0.0500   0.0340    0.0000    0.1060
  S2         0.1500   0.0779    0.0218    0.2782
  S3         0.3500   0.0475    0.2719    0.4281
  S4         0.8500   0.0252    0.8086    0.8914
  S5         0.4500   0.0637    0.3452    0.5548
  S6         0.2000   0.0625    0.0972    0.3028
  S7         0.9500   0.0126    0.9293    0.9707
  S8         0.0100   0.0031    0.0048    0.0152
  Leak       0.0200   0.0139    0.0000    0.0429

[phase1] Parameter store saved → params/param_store.json  (version: 20260221_175538)
S2 after conjugate update:
  mean=0.2000  std=0.0718  (was expert_mean=0.1500)



In [6]:
"""
phase2_simulate_validate.py — Prior Predictive Simulation & Validation
=======================================================================
GOAL: Before deploying anything, verify that the expert priors produce
      sensible risk score distributions. This catches misconfigured
      priors BEFORE they pollute production.

What this replaces from the original code:
  - PyMC sample_prior_predictive → pure NumPy MC (10–100x faster)
  - Static golden_weights.json   → already handled by param_store

Why Monte Carlo instead of PyMC here?
  - No posterior inference is needed (no labeled outcome data)
  - Beta samples are trivially drawn with numpy
  - 10,000 samples runs in <100ms vs ~30s for PyMC compilation

Outputs:
  - Console summary table
  - validation/simulation_results.json
  - plots saved to validation/plots/ (optional, requires matplotlib)

Run:
    python phase2_simulate_validate.py
"""

'\nphase2_simulate_validate.py — Prior Predictive Simulation & Validation\n=======================================================================\nGOAL: Before deploying anything, verify that the expert priors produce\n      sensible risk score distributions. This catches misconfigured\n      priors BEFORE they pollute production.\n\nWhat this replaces from the original code:\n  - PyMC sample_prior_predictive → pure NumPy MC (10–100x faster)\n  - Static golden_weights.json   → already handled by param_store\n\nWhy Monte Carlo instead of PyMC here?\n  - No posterior inference is needed (no labeled outcome data)\n  - Beta samples are trivially drawn with numpy\n  - 10,000 samples runs in <100ms vs ~30s for PyMC compilation\n\nOutputs:\n  - Console summary table\n  - validation/simulation_results.json\n  - plots saved to validation/plots/ (optional, requires matplotlib)\n\nRun:\n    python phase2_simulate_validate.py\n'

In [7]:
warnings.filterwarnings("ignore")

N_MC_SAMPLES = 10_000
PARAM_STORE_PATH = "params/param_store.json"
OUTPUT_DIR = "validation"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")


# ---------------------------------------------------------------------------
# Core MC Simulation
# ---------------------------------------------------------------------------
def draw_beta_samples(alpha: float, beta_val: float, n: int) -> np.ndarray:
    return np.random.beta(alpha, beta_val, size=n)


def simulate_risk_scores(
    store: dict,
    match_matrix: np.ndarray,   # shape (n_records, n_sources)
    n_mc: int = N_MC_SAMPLES,
    rng: np.random.Generator | None = None,
) -> dict:
    """
    For each record, draw n_mc samples of (p_i, p_leak) from their Beta priors
    and compute the Noisy-OR risk score.  Returns percentile summaries.

    match_matrix[r, i] = 1 if record r matched source i, else 0.

    Returns dict with keys:
      risk_mean, risk_std, risk_p10, risk_p50, risk_p90  — shape (n_records,)
      p_i_samples    — shape (n_mc, n_sources)
      p_leak_samples — shape (n_mc,)
    """
    if rng is None:
        rng = np.random.default_rng(42)

    source_order = store["source_order"]
    n_sources = len(source_order)
    n_records = match_matrix.shape[0]

    # Draw MC samples for each source probability
    p_i_samples = np.column_stack([
        rng.beta(store["sources"][s]["alpha"], store["sources"][s]["beta"], n_mc)
        for s in source_order
    ])  # (n_mc, n_sources)

    p_leak_samples = rng.beta(
        store["leak"]["alpha"], store["leak"]["beta"], n_mc
    )  # (n_mc,)

    # Noisy-OR for every (record × mc_sample) pair
    # log(prob_fail) = log(1-p_leak) + sum_i [ X_i * log(1-p_i) ]
    # Shape broadcast: (n_mc, n_sources) against (n_records, n_sources)
    log_one_minus_p = np.log1p(-p_i_samples)          # (n_mc, n_sources)
    log_p_leak_fail = np.log1p(-p_leak_samples)        # (n_mc,)

    # risk_mc[r, k] = risk score for record r, sample k
    risk_mc = np.zeros((n_records, n_mc), dtype=np.float32)
    for r in range(n_records):
        # (n_mc,): sum of source contributions for this record
        source_log_fail = match_matrix[r] @ log_one_minus_p.T   # (n_mc,)
        log_fail_total  = log_p_leak_fail + source_log_fail
        risk_mc[r]      = 1.0 - np.exp(log_fail_total)

    return {
        "risk_mean":   risk_mc.mean(axis=1).astype(float),
        "risk_std":    risk_mc.std(axis=1).astype(float),
        "risk_p10":    np.percentile(risk_mc, 10, axis=1).astype(float),
        "risk_p50":    np.percentile(risk_mc, 50, axis=1).astype(float),
        "risk_p90":    np.percentile(risk_mc, 90, axis=1).astype(float),
        "p_i_samples": p_i_samples,
        "p_leak_samples": p_leak_samples,
        "risk_mc":     risk_mc,
    }


# ---------------------------------------------------------------------------
# Point-estimate driver attribution (used in API scorer too)
# ---------------------------------------------------------------------------
def compute_impacts(
    match_vector: np.ndarray,   # (n_sources,)
    p_means: np.ndarray,        # (n_sources,)
    p_leak_mean: float,
    risk_score: float,
    log_fail_total: float,
) -> dict[str, float]:
    """
    Marginal impact of each active source:
      impact_i = risk_score - risk_without_i
    Where risk_without_i removes source i's log contribution.
    """
    impacts = {}
    log_one_minus_p = np.log1p(-p_means)
    for i, x in enumerate(match_vector):
        if x == 1:
            log_fail_without_i = log_fail_total - log_one_minus_p[i]
            risk_without_i = 1.0 - np.exp(log_fail_without_i)
            impacts[f"S{i+1}"] = round(float(risk_score - risk_without_i), 6)
    return impacts


# ---------------------------------------------------------------------------
# Validation checks
# ---------------------------------------------------------------------------
def validate_priors(store: dict, n_mc: int = N_MC_SAMPLES) -> dict:
    """
    Sanity checks on the prior specification:
      1. No source has near-certain risk (mean > 0.8) — would dominate everything
      2. Leak probability is low
      3. A zero-match record's baseline risk is reasonable
      4. A full-match record doesn't saturate above ~0.999
    """
    rng = np.random.default_rng(0)
    source_order = store["source_order"]
    n_sources = len(source_order)

    checks = {}

    # Check 1: Individual source means
    high_risk_sources = [
        s for s in source_order if store["sources"][s]["mean"] > 0.8
    ]
    checks["no_dominant_source"] = {
        "pass": len(high_risk_sources) == 0,
        "detail": f"Sources with mean>0.8: {high_risk_sources or 'none'}",
    }

    # Check 2: Leak is small
    leak_mean = store["leak"]["mean"]
    checks["leak_is_small"] = {
        "pass": leak_mean < 0.10,
        "detail": f"Leak mean = {leak_mean:.4f} (should be <0.10)",
    }

    # Check 3: Baseline (zero matches) risk
    zero_match = np.zeros((1, n_sources))
    res_zero = simulate_risk_scores(store, zero_match, n_mc=n_mc, rng=rng)
    baseline_p50 = float(res_zero["risk_p50"][0])
    checks["baseline_risk"] = {
        "pass": baseline_p50 < 0.15,
        "detail": f"Zero-match median risk = {baseline_p50:.4f} (should be <0.15)",
    }

    # Check 4: Full-match saturation
    full_match = np.ones((1, n_sources))
    res_full = simulate_risk_scores(store, full_match, n_mc=n_mc, rng=rng)
    full_p50 = float(res_full["risk_p50"][0])
    checks["full_match_saturates"] = {
        "pass": full_p50 > 0.80,
        "detail": f"Full-match median risk = {full_p50:.4f} (should be >0.80)",
    }

    passed = sum(c["pass"] for c in checks.values())
    checks["_summary"] = {
        "passed": passed,
        "total":  len([k for k in checks if not k.startswith("_")]),
    }
    return checks


def print_validation(checks: dict) -> None:
    print("\n--- Prior Validation ---")
    for name, c in checks.items():
        if name.startswith("_"):
            continue
        status = "✓ PASS" if c["pass"] else "✗ FAIL"
        print(f"  {status}  {name}: {c['detail']}")
    s = checks["_summary"]
    print(f"\n  {s['passed']}/{s['total']} checks passed\n")


# ---------------------------------------------------------------------------
# Source sensitivity analysis
# ---------------------------------------------------------------------------
def source_sensitivity(store: dict, n_mc: int = 2_000) -> dict[str, float]:
    """
    For each source, estimate its average marginal impact by comparing
    a single-source match vs zero match, averaged over MC samples.
    """
    rng = np.random.default_rng(1)
    source_order = store["source_order"]
    n_sources = len(source_order)

    zero = np.zeros((1, n_sources))
    res_zero = simulate_risk_scores(store, zero, n_mc=n_mc, rng=rng)
    base_risk = res_zero["risk_mean"][0]

    sensitivity = {}
    for i, s in enumerate(source_order):
        single = np.zeros((1, n_sources))
        single[0, i] = 1
        res = simulate_risk_scores(store, single, n_mc=n_mc, rng=rng)
        sensitivity[s] = round(float(res["risk_mean"][0] - base_risk), 6)

    return dict(sorted(sensitivity.items(), key=lambda x: -x[1]))


# ---------------------------------------------------------------------------
# Save results
# ---------------------------------------------------------------------------
def save_validation_results(
    checks: dict,
    sensitivity: dict,
    path: str = "validation/validation_results.json",
) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({"validation_checks": checks, "source_sensitivity": sensitivity}, f, indent=4)
    print(f"[phase2] Validation results saved → {path}")

In [8]:
# ---------------------------------------------------------------------------
# Optional plots
# ---------------------------------------------------------------------------
def save_plots(store: dict, n_mc: int = N_MC_SAMPLES) -> None:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("[phase2] matplotlib not available, skipping plots.")
        return

    os.makedirs(PLOT_DIR, exist_ok=True)
    rng = np.random.default_rng(99)
    source_order = store["source_order"]
    n_sources = len(source_order)

    # --- Plot 1: Prior distributions (Beta PDFs) ---
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    axes = axes.flatten()
    x = np.linspace(0.001, 0.999, 500)
    for i, s in enumerate(source_order):
        from scipy.stats import beta as sp_beta
        a, b = store["sources"][s]["alpha"], store["sources"][s]["beta"]
        axes[i].plot(x, sp_beta.pdf(x, a, b), color="steelblue", lw=2)
        axes[i].axvline(store["sources"][s]["mean"], color="red", ls="--", lw=1.2)
        axes[i].set_title(f"{s}  (μ={store['sources'][s]['mean']:.3f})", fontsize=10)
        axes[i].set_xlabel("P(risk | match)")
        axes[i].set_xlim(0, 1)
    plt.suptitle("Expert Prior Distributions per Source (red = mean)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "prior_distributions.png"), dpi=120)
    plt.close()

    # --- Plot 2: Risk score distribution for a random match matrix ---
    n_test = 500
    match_matrix = np.random.binomial(1, 0.25, (n_test, n_sources))
    res = simulate_risk_scores(store, match_matrix, n_mc=n_mc, rng=rng)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(res["risk_p50"], bins=40, color="teal", edgecolor="white", alpha=0.8)
    ax.set_xlabel("Median Risk Score (p50 across MC samples)")
    ax.set_ylabel("Count")
    ax.set_title("Simulated Risk Score Distribution (500 random records, 25% match rate)")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "risk_score_distribution.png"), dpi=120)
    plt.close()

    # --- Plot 3: Source sensitivity ---
    sens = source_sensitivity(store)
    fig, ax = plt.subplots(figsize=(7, 4))
    sources_sorted = list(sens.keys())
    values = [sens[s] for s in sources_sorted]
    ax.barh(sources_sorted, values, color="steelblue", edgecolor="white")
    ax.set_xlabel("Average Marginal Risk Increase (single match vs zero)")
    ax.set_title("Source Sensitivity Analysis")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "source_sensitivity.png"), dpi=120)
    plt.close()

    print(f"[phase2] Plots saved → {PLOT_DIR}/")

In [9]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    store = load_param_store(PARAM_STORE_PATH)

    print("\n[phase2] Running prior validation...")
    checks = validate_priors(store)
    print_validation(checks)

    print("[phase2] Computing source sensitivity...")
    sensitivity = source_sensitivity(store)
    print("  Source sensitivity (avg marginal impact of a single match):")
    for s, v in sensitivity.items():
        bar = "█" * int(v * 200)
        print(f"    {s}: {v:.4f}  {bar}")

    save_validation_results(checks, sensitivity)
    save_plots(store)

    print("\n[phase2] Complete. Proceed to phase3_scorer.py")


[phase2] Running prior validation...

--- Prior Validation ---
  ✗ FAIL  no_dominant_source: Sources with mean>0.8: ['S4', 'S7']
  ✓ PASS  leak_is_small: Leak mean = 0.0200 (should be <0.10)
  ✓ PASS  baseline_risk: Zero-match median risk = 0.0173 (should be <0.15)
  ✓ PASS  full_match_saturates: Full-match median risk = 0.9984 (should be >0.80)

  3/4 checks passed

[phase2] Computing source sensitivity...
  Source sensitivity (avg marginal impact of a single match):
    S7: 0.9311  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    S4: 0.8328  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    S5: 0.4414  ████████████████████████████████████████████████████████████████████████████████████████
    S3: 0.3447  ███████████████████

In [10]:
"""
phase3_scorer.py — Core Vectorized Scorer
==========================================
Single source of truth for all scoring logic.
Both the API and batch pipeline import from here — no duplication.

Key design decisions:
  1. Inputs are always 2D numpy arrays → works for n=1 (API) or n=1M (batch)
  2. Uncertainty via MC sampling from Beta priors (fast, accurate)
  3. Driver attribution is fully vectorized (no Python loops over sources)
  4. Param store is loaded once and cached (pass as argument for testability)

Public API:
    score(match_matrix, store, n_mc) → pd.DataFrame
    score_single(match_vector, store, n_mc) → dict
"""

'\nphase3_scorer.py — Core Vectorized Scorer\n==========================================\nSingle source of truth for all scoring logic.\nBoth the API and batch pipeline import from here — no duplication.\n\nKey design decisions:\n  1. Inputs are always 2D numpy arrays → works for n=1 (API) or n=1M (batch)\n  2. Uncertainty via MC sampling from Beta priors (fast, accurate)\n  3. Driver attribution is fully vectorized (no Python loops over sources)\n  4. Param store is loaded once and cached (pass as argument for testability)\n\nPublic API:\n    score(match_matrix, store, n_mc) → pd.DataFrame\n    score_single(match_vector, store, n_mc) → dict\n'

In [11]:
import json
import numpy as np
import pandas as pd
from typing import Any

In [12]:
PARAM_STORE_PATH = "params/param_store.json"
DEFAULT_N_MC = 1_000   # Fast for API; increase for higher-fidelity uncertainty
BATCH_N_MC   = 500     # Slightly lower for large batch runs (speed vs accuracy tradeoff)

In [13]:

# ---------------------------------------------------------------------------
# Param loading (cached at module level for API reuse)
# ---------------------------------------------------------------------------
_cached_store: dict | None = None

def load_store(path: str = PARAM_STORE_PATH) -> dict:
    global _cached_store
    if _cached_store is None:
        with open(path) as f:
            _cached_store = json.load(f)
    return _cached_store

def reload_store(path: str = PARAM_STORE_PATH) -> dict:
    """Force reload — call when params have been updated."""
    global _cached_store
    with open(path) as f:
        _cached_store = json.load(f)
    return _cached_store


# ---------------------------------------------------------------------------
# Pre-compute parameter arrays from store (fast repeated access)
# ---------------------------------------------------------------------------
def _extract_params(store: dict) -> tuple[np.ndarray, np.ndarray, float, float, list[str]]:
    """
    Returns (alphas, betas, leak_alpha, leak_beta, source_order).
    All as numpy arrays for vectorized sampling.
    """
    source_order = store["source_order"]
    alphas = np.array([store["sources"][s]["alpha"] for s in source_order])
    betas  = np.array([store["sources"][s]["beta"]  for s in source_order])
    return (
        alphas,
        betas,
        store["leak"]["alpha"],
        store["leak"]["beta"],
        source_order,
    )


# ---------------------------------------------------------------------------
# Core scoring engine
# ---------------------------------------------------------------------------
def _score_core(
    match_matrix: np.ndarray,    # (n_records, n_sources), values in {0, 1}
    alphas:       np.ndarray,    # (n_sources,)
    betas:        np.ndarray,    # (n_sources,)
    leak_alpha:   float,
    leak_beta:    float,
    n_mc:         int,
    rng:          np.random.Generator,
) -> dict[str, np.ndarray]:
    """
    Pure numpy scoring with MC uncertainty.

    Returns:
      risk_point  (n_records,)  — point estimate using posterior means
      risk_p50    (n_records,)  — median over MC samples
      risk_p10    (n_records,)  — 10th percentile (lower credible bound)
      risk_p90    (n_records,)  — 90th percentile (upper credible bound)
      risk_std    (n_records,)  — std over MC samples
      impact_matrix (n_records, n_sources) — marginal impact per source
    """
    n_records, n_sources = match_matrix.shape

    # --- Point estimate (posterior means — instantaneous, no MC needed) ---
    p_means     = alphas / (alphas + betas)
    p_leak_mean = leak_alpha / (leak_alpha + leak_beta)

    log_one_minus_p_means  = np.log1p(-p_means)        # (n_sources,)
    log_p_leak_fail_mean   = np.log1p(-p_leak_mean)     # scalar

    # (n_records,)
    source_log_fail_point = match_matrix @ log_one_minus_p_means
    log_fail_point        = log_p_leak_fail_mean + source_log_fail_point
    risk_point            = 1.0 - np.exp(log_fail_point)

    # --- Vectorized driver attribution (point estimate) ---
    # impact[r, i] = risk_point[r] - risk_without_source_i[r]
    # risk_without_i = 1 - exp(log_fail_total - X[r,i]*log(1-p_i))
    # Broadcast: (n_records, n_sources)
    log_one_minus_p_broadcast = log_one_minus_p_means[np.newaxis, :]   # (1, n_sources)
    log_fail_without_i = (
        log_fail_point[:, np.newaxis]                                    # (n_records, 1)
        - match_matrix * log_one_minus_p_broadcast                       # (n_records, n_sources)
    )
    risk_without_i  = 1.0 - np.exp(log_fail_without_i)                  # (n_records, n_sources)
    impact_matrix   = risk_point[:, np.newaxis] - risk_without_i         # (n_records, n_sources)
    impact_matrix  *= match_matrix                                        # zero out non-matches

    # --- MC uncertainty (draws from Beta priors) ---
    p_i_mc    = rng.beta(alphas, betas, size=(n_mc, n_sources))   # (n_mc, n_sources)
    p_leak_mc = rng.beta(leak_alpha, leak_beta, size=n_mc)         # (n_mc,)

    log_1mp_mc       = np.log1p(-p_i_mc)     # (n_mc, n_sources)
    log_leak_fail_mc = np.log1p(-p_leak_mc)  # (n_mc,)

    # risk_mc[r, k] — broadcast (n_records, n_sources) × (n_mc, n_sources).T
    # = (n_records, n_mc)
    source_log_fail_mc = match_matrix @ log_1mp_mc.T    # (n_records, n_mc)
    log_fail_mc        = log_leak_fail_mc[np.newaxis, :] + source_log_fail_mc
    risk_mc            = (1.0 - np.exp(log_fail_mc)).astype(np.float32)

    return {
        "risk_point":    risk_point,
        "risk_p10":      np.percentile(risk_mc, 10, axis=1),
        "risk_p50":      np.percentile(risk_mc, 50, axis=1),
        "risk_p90":      np.percentile(risk_mc, 90, axis=1),
        "risk_std":      risk_mc.std(axis=1),
        "impact_matrix": impact_matrix,
    }


# ---------------------------------------------------------------------------
# Public scoring functions
# ---------------------------------------------------------------------------
def score(
    match_matrix: np.ndarray,
    store:   dict | None = None,
    n_mc:    int  = DEFAULT_N_MC,
    seed:    int  = 0,
    source_names: list[str] | None = None,
) -> pd.DataFrame:
    """
    Score a batch of records.

    Parameters
    ----------
    match_matrix : np.ndarray, shape (n_records, n_sources), dtype int/float
        Binary match indicators. Columns must align with store["source_order"].
    store : dict, optional
        Parameter store. Loaded from disk if None.
    n_mc : int
        Number of Monte Carlo samples for uncertainty estimation.
    seed : int
        RNG seed for reproducibility.
    source_names : list[str], optional
        Override column names for the impact columns.

    Returns
    -------
    pd.DataFrame with columns:
        risk_point, risk_p50, risk_p10, risk_p90, risk_std,
        primary_driver, impact_width,
        S1_impact, S2_impact, ..., S8_impact
    """
    if store is None:
        store = load_store()

    rng = np.random.default_rng(seed)
    alphas, betas, leak_alpha, leak_beta, source_order = _extract_params(store)

    if source_names is None:
        source_names = source_order

    match_matrix = np.asarray(match_matrix, dtype=np.float64)
    assert match_matrix.shape[1] == len(source_order), (
        f"match_matrix has {match_matrix.shape[1]} columns but store has {len(source_order)} sources"
    )

    result = _score_core(match_matrix, alphas, betas, leak_alpha, leak_beta, n_mc, rng)

    n_records = match_matrix.shape[0]
    df = pd.DataFrame({
        "risk_point": np.round(result["risk_point"], 4),
        "risk_p50":   np.round(result["risk_p50"],   4),
        "risk_p10":   np.round(result["risk_p10"],   4),
        "risk_p90":   np.round(result["risk_p90"],   4),
        "risk_std":   np.round(result["risk_std"],   4),
        "impact_width": np.round(result["risk_p90"] - result["risk_p10"], 4),
    })

    # Driver attribution columns
    impact_matrix = result["impact_matrix"]
    for i, s in enumerate(source_names):
        df[f"{s}_impact"] = np.round(impact_matrix[:, i], 6)

    # Primary driver (source with highest marginal impact; "Leak" if all zero)
    max_idx     = np.argmax(impact_matrix, axis=1)
    max_val     = impact_matrix[np.arange(n_records), max_idx]
    primary     = np.where(max_val > 0, [source_names[i] for i in max_idx], "Leak")
    df["primary_driver"] = primary

    # Matched sources (human-readable string)
    df["matched_sources"] = [
        ", ".join(s for j, s in enumerate(source_names) if match_matrix[r, j] == 1) or "none"
        for r in range(n_records)
    ]

    return df


def score_single(
    match_vector: np.ndarray,    # (n_sources,) or list
    store:   dict | None = None,
    n_mc:    int  = DEFAULT_N_MC,
    seed:    int  = 0,
) -> dict[str, Any]:
    """
    Score a single record. Wrapper around score() for API use.

    Returns a plain dict (JSON-serializable).
    """
    if store is None:
        store = load_store()

    mv = np.asarray(match_vector, dtype=np.float64).reshape(1, -1)
    df = score(mv, store=store, n_mc=n_mc, seed=seed)
    row = df.iloc[0].to_dict()

    # Build a clean impact sub-dict
    source_order = store["source_order"]
    impacts = {s: row.pop(f"{s}_impact", 0.0) for s in source_order}
    row["impacts"] = impacts

    return row



In [14]:
# ---------------------------------------------------------------------------
# Quick self-test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    from phase1_priors import load_param_store, SOURCE_SPECS, LEAK_SPEC, build_param_store, save_param_store
    import os

    # Build store if it doesn't exist yet
    if not os.path.exists(PARAM_STORE_PATH):
        store = build_param_store(SOURCE_SPECS, LEAK_SPEC)
        save_param_store(store)

    store = load_param_store(PARAM_STORE_PATH)

    print("\n=== Single Record Test ===")
    test_vec = np.array([1, 0, 0, 1, 0, 0, 1, 0])  # S1, S4, S7 matched
    result = score_single(test_vec, store=store, n_mc=2_000)
    print(f"  Matches: S1, S4, S7")
    print(f"  risk_point : {result['risk_point']}")
    print(f"  risk_p50   : {result['risk_p50']}")
    print(f"  risk_p10   : {result['risk_p10']}  →  risk_p90: {result['risk_p90']}")
    print(f"  impact_width: {result['impact_width']} (credible interval width)")
    print(f"  primary_driver: {result['primary_driver']}")
    print(f"  impacts: {result['impacts']}")

    print("\n=== Batch Test (100 records) ===")
    np.random.seed(42)
    batch_matrix = np.random.binomial(1, 0.25, (100, 8))
    df = score(batch_matrix, store=store, n_mc=500)
    print(df[["risk_point", "risk_p50", "risk_p10", "risk_p90", "primary_driver"]].describe())
    print(f"\n  Primary driver distribution:\n{df['primary_driver'].value_counts()}")
    print("\n[phase3] Scorer tests passed.")


=== Single Record Test ===
  Matches: S1, S4, S7
  risk_point : 0.993
  risk_p50   : 0.9933000206947327
  risk_p10   : 0.9904000163078308  →  risk_p90: 0.9955000281333923
  impact_width: 0.005100000184029341 (credible interval width)
  primary_driver: S7
  impacts: {'S1': 0.000367, 'S2': 0.0, 'S3': 0.0, 'S4': 0.039568, 'S5': 0.0, 'S6': 0.0, 'S7': 0.132668, 'S8': 0.0}

=== Batch Test (100 records) ===
       risk_point    risk_p50    risk_p10    risk_p90
count  100.000000  100.000000  100.000000  100.000000
mean     0.565308    0.564395    0.530466    0.604140
std      0.387595    0.389982    0.401272    0.373129
min      0.020000    0.016600    0.005400    0.038500
25%      0.200275    0.194150    0.109675    0.289375
50%      0.555600    0.560750    0.473150    0.651550
75%      0.951000    0.952000    0.936400    0.964700
max      0.997800    0.998000    0.996900    0.998600

  Primary driver distribution:
primary_driver
S7      27
S4      20
Leak    13
S6      10
S5       9
S3     